# Proyecto final: Monitorización de la calidad del aire en ciudades inteligencias

Autor: Adrián Robles Arques

## Fases del proyecto

### Fase 1: Preparación del modelo de ML de Regresión Logística

Reutiliza el modelo de Regresión Logística utilizado en el Sprint 3 como paso inicial para la fase de streaming. En este caso, adaptaremos el modelo para predecir la calidad del aire en función de diversos parámetros ambientales.

 

### Fase 2: Preparación de los datos

* Obtención de Datos Ficticios: Los estudiantes deben crear datos ficticios que simulan mediciones de calidad del aire, como concentración de CO2, partículas PM2.5, temperatura, humedad, etc.
* División de Datos: Dividir estos datos ficticios en 20 archivos CSV de tamaño similar.
* Almacenamiento de Datos: Guardar cada parte de los datos en un directorio que será la fuente del stream de datos.

 

### Fase 3: Creación de la fuente de streaming

* Creación de la Fuente: Utilizar el método readStream() para crear la fuente de datos para el stream, leyendo los datos desde el directorio mencionado anteriormente.
* Transformación con el Modelo: Aplicar el método transform() sobre el modelo de Regresión Logística previamente entrenado, pasando como parámetro el stream de fuente creado en el paso anterior.

* Visualización del Stream: Utilizar el método display() para visualizar el stream de datos resultante.

 

### Fase 4: Implementación de una Consulta adicional

Crear una segunda consulta sobre el stream resultante del paso anterior:

* Utilizar el método writeStream() con un modo de salida “append”, formato “memory” y nombre “calidadAireClassification”.

 

### Fase 5: Evaluación del Modelo en streaming

* Creación del Evaluador: Crear un objeto evaluador del tipo MulticlassClassificationEvaluator para obtener la métrica de "accuracy".
* Generación de Dataframe: A partir de la consulta “calidadAireClassification”, generar un DataFrame utilizando spark.sql().
* Evaluación del Modelo: Pasar este DataFrame resultante como parámetro al evaluador utilizando el método evaluate().
* Impresión de la Métrica: Imprimir la métrica de "accuracy" resultante para el modelo.


## Fase 1: Carga y revisión de los datos

En esta fase vamos a realizar una primera inspección de los datos, tanto de los diferentes sensores en tierra donde se evalúa el índice de calidad del aire para cada uno de los contaminantes considerados, así como los datos meteorológicos proporcionados por Meteosat para la ciudad de Valencia.

De este modo, para evaluar cada uno de los índices individuales tomaremos en consideración no solo valores previos, como sería usual en una serie temporal, si no también enriquecer esta fuente de datos con información meteorológica.

In [ ]:
# Instalamos la libreria polars para manejo de datos
!pip install polars

  Using cached polars-1.31.0-cp39-abi3-win_amd64.whl.metadata (15 kB)
Using cached polars-1.31.0-cp39-abi3-win_amd64.whl (35.2 MB)


In [1]:
# Importamos librerías necesarias
import numpy as np
import polars as pl # Voy a incluir polars para manejar los datos

In [2]:
# Vamos a crear un DataFrame para los datos climáticos

# Importamos los datos climáticos
datos_clima_2024 = 'C:\\Users\\demad\\Desktop\\Test\\DataScienceIEBS\\Bloque 7\\Spark\\Proyecto final\\Datos calidad aire\\meteosat_val_2024.csv'
df_clima = pl.read_csv(datos_clima_2024)

# Visualizamos un fragmento del DataFrame
df_clima.head()

date,tavg,tmin,tmax,prcp,snow,wdir,wspd,wpgt,pres,tsun
str,f64,f64,f64,f64,str,str,f64,str,f64,str
"""2024-01-01 00:00:00""",11.1,9.8,19.0,0.0,null,null,11.8,null,1019.7,null
"""2024-01-02 00:00:00""",13.0,7.8,18.9,0.0,null,null,24.3,null,1019.8,null
"""2024-01-03 00:00:00""",17.4,15.0,23.0,0.0,null,null,26.4,null,1017.0,null
"""2024-01-04 00:00:00""",14.8,14.5,20.6,0.6,null,null,12.1,null,1014.8,null
"""2024-01-05 00:00:00""",13.1,12.4,18.4,0.0,null,null,22.7,null,1006.0,null


Aquí vemos que los datos climáticos que tenemos son:
* Temperatura promedio (C)
* Temperatura mínima (C)
* Temperatura máxima (C)
* Precipitación acumulada (mm)
* Profundidad de la nieve
* Dirección del viento
* Velocidad del viento (km/h)
* Ráfaga de viento
* Presión del aire (hPa)
* Duración del sol

Como vemos, varias de las columnas no son necesarios o están vacías, por lo que vamos a quedarnos solo con las que nos interesan.
* Temperatura promedio (C)
* Temperatura mínima (C)
* Temperatura máxima (C)
* Precipitación acumulada (mm)
* Velocidad del viento (km/h)
* Presión del aire (hPa)

In [3]:
# Vamos a modificar el DataFrame para que tenga las columnas adecuadas
df_clima = df_clima.drop(['snow', 'wdir', 'wpgt', 'tsun'])

In [4]:
# También vamos a renombrar las columnas para que sean más descriptivas
df_clima = df_clima.rename({
'date': 'fecha',
'tavg': 'temperatura_media',
'tmin': 'temperatura_minima',
'tmax': 'temperatura_maxima',
'prcp': 'precipitacion',
'wspd': 'velocidad_viento',
'pres': 'presion_atmosferica'
})

In [5]:
# Mostramos nuevamente el DataFrame para ver los cambios
df_clima.head(10)

fecha,temperatura_media,temperatura_minima,temperatura_maxima,precipitacion,velocidad_viento,presion_atmosferica
str,f64,f64,f64,f64,f64,f64
"""2024-01-01 00:00:00""",11.1,9.8,19.0,0.0,11.8,1019.7
"""2024-01-02 00:00:00""",13.0,7.8,18.9,0.0,24.3,1019.8
"""2024-01-03 00:00:00""",17.4,15.0,23.0,0.0,26.4,1017.0
"""2024-01-04 00:00:00""",14.8,14.5,20.6,0.6,12.1,1014.8
"""2024-01-05 00:00:00""",13.1,12.4,18.4,0.0,22.7,1006.0
"""2024-01-06 00:00:00""",11.9,9.0,18.2,0.0,25.6,1013.3
"""2024-01-07 00:00:00""",10.8,8.3,16.6,0.0,30.0,1018.7
"""2024-01-08 00:00:00""",9.3,8.4,17.6,0.0,18.5,1018.5
"""2024-01-09 00:00:00""",9.5,7.6,15.6,0.0,8.6,1020.2


In [6]:
# Vamos a cargar los datos de calidad del aire
sensores = {
    'viver': 'C:\\Users\\demad\\Desktop\\Test\\DataScienceIEBS\\Bloque 7\\Spark\\Proyecto final\\Datos calidad aire\\sensor viver.csv',
    'moli del sol': 'C:\\Users\\demad\\Desktop\\Test\\DataScienceIEBS\\Bloque 7\\Spark\\Proyecto final\\Datos calidad aire\\sensor moli del sol.csv',
    'pista de silla': 'C:\\Users\\demad\\Desktop\\Test\\DataScienceIEBS\\Bloque 7\\Spark\\Proyecto final\\Datos calidad aire\\sensor pista de silla.csv',
    'politècnic': 'C:\\Users\\demad\\Desktop\\Test\\DataScienceIEBS\\Bloque 7\\Spark\\Proyecto final\\Datos calidad aire\\sensor politecnic.csv',
    'quart de poblet': 'C:\\Users\\demad\\Desktop\\Test\\DataScienceIEBS\\Bloque 7\\Spark\\Proyecto final\\Datos calidad aire\\sensor quart de poblet.csv'
}

# Cargamos los datos de cada sensor y los almacenamos en un diccionario
sensor_viver = pl.read_csv(sensores['viver'])
sensor_moli_del_sol = pl.read_csv(sensores['moli del sol'])
sensor_pista_de_silla = pl.read_csv(sensores['pista de silla'])
sensor_politecnic = pl.read_csv(sensores['politècnic'])
sensor_quart_de_poblet = pl.read_csv(sensores['quart de poblet'])

In [7]:
# Vamos a explorar los datos de un sensor como ejemplo
sensor_viver.head()

date,pm25,pm10,o3,no2,so2,co
str,str,str,str,str,str,str
"""2025/6/1""",""" 18""",""" 6""",""" 40""",""" 4""",""" 1""",""" """
"""2025/6/2""",""" 15""",""" 7""",""" 41""",""" 2""",""" 1""",""" """
"""2025/6/3""",""" 19""",""" 9""",""" 43""",""" 3""",""" 1""",""" """
"""2025/6/4""",""" 32""",""" 12""",""" 45""",""" 3""",""" 1""",""" """
"""2025/6/5""",""" 38""",""" 12""",""" 48""",""" 3""",""" 1""",""" """


Como puede verse, los datos de los sensores ya están convertidos para mostrar el Índice calculado para cada uno de los contaminantes. También se aprecia que hay algunos valores que están en blanco, por lo que habrá que imputarlos. Además, los datos se han cargado en formato cadena, lo que requerirá una transformación. Dado que los datos son diaios, vamos a modificar las fechas de Meteosat para que dejen de mostrar la hora, puesto que es un dato irrelevante.